In [10]:
import numpy as np
import polars as pl
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit

In [11]:
DATA_PATH = "raw/processed_gtfs/baseline_dataset.parquet"
TARGET = "travel_time"
SEQ_ID_COLS = ["trip_id", "start_date"]
ORDER_COL = "stop_sequence"
 
CATEGORICAL = [
    "route_id", "direction_id", "shape_id", "service_id",
    "is_raining", "is_snowing", "is_fog", "is_weekend",
    "is_federal_holiday", "is_school_day", "has_major_event",
    "is_peak", "weathercode",
]
NUMERIC = [
    "stop_sequence", "trip_progress",
    "hour", "weekday", "month",
    "scheduled_arrival", "scheduled_departure", "scheduled_segment_time",
    "stop_lat", "stop_lon", "latitude", "longitude", "bearing",
    "temperature_c", "precipitation_mm", "snowfall_cm", "windspeed_kmh",
    "segment_length", "scheduled_segment_speed_mps",
    "upstream_delay_seconds", "speed_mps", "headway_seconds",
    "ridership", "transfers",
]
 
HIDDEN_SIZE = 256
NUM_LAYERS = 3
EMB_DIM_CAP = 150
DROPOUT = 0.1
BATCH_SIZE = 64
LR = 3e-3
MAX_EPOCHS = 100
PATIENCE = 12
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
torch.manual_seed(42)
np.random.seed(42)
 
 

In [12]:
df = pl.read_parquet(DATA_PATH)
print(f"Loaded {df.shape}")
 
df = df.with_columns(
    (pl.col("trip_id") + "_" + pl.col("start_date").cast(pl.Utf8)).alias("_seq_id")
)
 
groups = df["_seq_id"].to_numpy()
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=groups))
train_df = df[train_idx]
test_df = df[test_idx]
print(f"train rows: {train_df.height:,} | test rows: {test_df.height:,}")
print(f"train trips: {train_df['_seq_id'].n_unique():,} | test trips: {test_df['_seq_id'].n_unique():,}")

Loaded (5596768, 40)
train rows: 4,478,560 | test rows: 1,118,208
train trips: 96,308 | test trips: 24,077


In [13]:
cat_maps = {}
cat_cardinalities = []
for col in CATEGORICAL:
    uniques = train_df[col].unique().sort().to_list()
    cat_maps[col] = {v: i + 1 for i, v in enumerate(uniques)}
    cat_cardinalities.append(len(uniques) + 1)
 
 
def encode_categoricals(d: pl.DataFrame) -> pl.DataFrame:
    exprs = []
    for col in CATEGORICAL:
        mapping = cat_maps[col]
        exprs.append(
            pl.col(col).cast(pl.Utf8).replace(mapping, default=0).cast(pl.Int64).alias(f"_{col}_code")
        )
    return d.with_columns(exprs)
 
 
train_df = encode_categoricals(train_df)
test_df = encode_categoricals(test_df)
CAT_CODE_COLS = [f"_{c}_code" for c in CATEGORICAL]

C:\Users\ishan\AppData\Local\Temp\ipykernel_19496\3771976203.py:14: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col(col).cast(pl.Utf8).replace(mapping, default=0).cast(pl.Int64).alias(f"_{col}_code")


In [14]:
num_means = {c: train_df[c].mean() for c in NUMERIC}
num_stds = {c: (train_df[c].std() or 1.0) for c in NUMERIC}
num_stds = {c: (s if s and s > 1e-8 else 1.0) for c, s in num_stds.items()}
 
 
def normalize_numeric(d: pl.DataFrame) -> pl.DataFrame:
    return d.with_columns([
        ((pl.col(c) - num_means[c]) / num_stds[c]).alias(c) for c in NUMERIC
    ])
 
 
train_df = normalize_numeric(train_df)
test_df = normalize_numeric(test_df)

In [15]:
def build_sequences(d: pl.DataFrame) -> list[dict]:
    d = d.sort(SEQ_ID_COLS + [ORDER_COL])
    sequences = []
    for _, group in d.group_by("_seq_id", maintain_order=True):
        cat = group.select(CAT_CODE_COLS).to_numpy()
        num = group.select(NUMERIC).to_numpy().astype(np.float32)
        target = group[TARGET].to_numpy().astype(np.float32)
        sequences.append({"cat": cat, "num": num, "target": target, "length": len(target)})
    return sequences
 
 
print("Building train sequences...")
train_sequences = build_sequences(train_df)
print("Building test sequences...")
test_sequences = build_sequences(test_df)
print(f"train sequences: {len(train_sequences):,} | test sequences: {len(test_sequences):,}")

Building train sequences...
Building test sequences...
train sequences: 96,308 | test sequences: 24,077


In [16]:
class TripSequenceDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences
 
    def __len__(self):
        return len(self.sequences)
 
    def __getitem__(self, idx):
        s = self.sequences[idx]
        return (
            torch.tensor(s["cat"], dtype=torch.long),
            torch.tensor(s["num"], dtype=torch.float32),
            torch.tensor(s["target"], dtype=torch.float32),
            s["length"],
        )
 
 
def collate(batch):
    cats, nums, targets, lengths = zip(*batch)
    lengths = torch.tensor(lengths, dtype=torch.long)
    cats_padded = pad_sequence(cats, batch_first=True, padding_value=0)
    nums_padded = pad_sequence(nums, batch_first=True, padding_value=0.0)
    targets_padded = pad_sequence(targets, batch_first=True, padding_value=0.0)
    return cats_padded, nums_padded, targets_padded, lengths
 
 
train_loader = DataLoader(TripSequenceDataset(train_sequences), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
test_loader = DataLoader(TripSequenceDataset(test_sequences), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

In [17]:
class GRUTravelTime(nn.Module):
    def __init__(self, cc, nn_c, hs, nl, dp):
        super().__init__()
        # Fast.ai dynamic embedding sizing rule (same as LSTM version)
        ed = [min(EMB_DIM_CAP, max(1, int(1.6 * (c ** 0.56)))) for c in cc]
 
        self.e = nn.ModuleList([
            nn.Embedding(c, d, padding_idx=0) for c, d in zip(cc, ed)
        ])
 
        ins = sum(ed) + nn_c
        self.g = nn.GRU(
            ins, hs, num_layers=nl, batch_first=True,
            dropout=dp if nl > 1 else 0.0,
        )
        self.h = nn.Sequential(
            nn.Linear(hs, hs),
            nn.ReLU(),
            nn.Dropout(dp),
            nn.Linear(hs, hs // 2),
            nn.ReLU(),
            nn.Dropout(dp),
            nn.Linear(hs // 2, 1),
        )
 
    def forward(self, c, n, l):
        em = [emb(c[:, :, i]) for i, emb in enumerate(self.e)]
        x = torch.cat(em + [n], dim=-1)
        p = pack_padded_sequence(x, l.cpu(), batch_first=True, enforce_sorted=False)
        po, _ = self.g(p)   # GRU returns (output, h_n) — no cell state, same unpack pattern
        o, _ = pad_packed_sequence(po, batch_first=True, total_length=x.size(1))
        return self.h(o).squeeze(-1)
 
 
def masked_mae(pred: torch.Tensor, target: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    mask = (torch.arange(pred.size(1), device=pred.device).unsqueeze(0) < lengths.unsqueeze(1).to(pred.device))
    return (torch.abs(pred - target) * mask).sum() / mask.sum().clamp(min=1)
 
 
def masked_mse(pred: torch.Tensor, target: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    mask = (torch.arange(pred.size(1), device=pred.device).unsqueeze(0) < lengths.unsqueeze(1).to(pred.device))
    return (((pred - target) ** 2) * mask).sum() / mask.sum().clamp(min=1)
 
 
model = GRUTravelTime(cat_cardinalities, len(NUMERIC), HIDDEN_SIZE, NUM_LAYERS, DROPOUT).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=3e-3,
    epochs=MAX_EPOCHS,
    steps_per_epoch=len(train_loader),
)
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"total parameters: {n_params:,}")

GRUTravelTime(
  (e): ModuleList(
    (0): Embedding(6, 4, padding_idx=0)
    (1): Embedding(3, 2, padding_idx=0)
    (2): Embedding(79, 18, padding_idx=0)
    (3): Embedding(49, 14, padding_idx=0)
    (4-5): 2 x Embedding(3, 2, padding_idx=0)
    (6): Embedding(2, 2, padding_idx=0)
    (7-11): 5 x Embedding(3, 2, padding_idx=0)
    (12): Embedding(14, 7, padding_idx=0)
  )
  (g): GRU(85, 256, num_layers=3, batch_first=True, dropout=0.1)
  (h): Sequential(
    (0): Linear(in_features=256, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.1, inplace=False)
    (6): Linear(in_features=128, out_features=1, bias=True)
  )
)
total parameters: 1,154,027


In [18]:
def run_epoch(loader, train: bool) -> float:
    model.train() if train else model.eval()
    total_mae, total_count = 0.0, 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for cat, num, target, lengths in loader:
            cat, num, target = cat.to(DEVICE), num.to(DEVICE), target.to(DEVICE)
            pred = model(cat, num, lengths)
            loss = masked_mse(pred, target, lengths)
            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()
                scheduler.step()
            mae = masked_mae(pred, target, lengths)
            n = lengths.sum().item()
            total_mae += mae.item() * n
            total_count += n
    return total_mae / total_count
 
 
best_val_mae = float("inf")
epochs_no_improve = 0
best_state = None
 
for epoch in range(1, MAX_EPOCHS + 1):
    train_mae = run_epoch(train_loader, train=True)
    val_mae = run_epoch(test_loader, train=False)
    print(f"epoch {epoch:3d} | train MAE {train_mae:7.2f}s | val MAE {val_mae:7.2f}s")
 
    if val_mae < best_val_mae - 1e-3:
        best_val_mae = val_mae
        epochs_no_improve = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch} (best val MAE {best_val_mae:.2f}s)")
            break
 
model.load_state_dict(best_state)

epoch   1 | train MAE   44.88s | val MAE   33.83s
epoch   2 | train MAE   33.18s | val MAE   31.99s
epoch   3 | train MAE   32.22s | val MAE   31.47s
epoch   4 | train MAE   31.67s | val MAE   30.76s
epoch   5 | train MAE   31.32s | val MAE   30.76s
epoch   6 | train MAE   31.09s | val MAE   30.67s
epoch   7 | train MAE   30.92s | val MAE   30.29s
epoch   8 | train MAE   30.77s | val MAE   30.40s
epoch   9 | train MAE   30.68s | val MAE   29.99s
epoch  10 | train MAE   30.62s | val MAE   30.21s
epoch  11 | train MAE   30.57s | val MAE   29.90s
epoch  12 | train MAE   30.57s | val MAE   31.34s
epoch  13 | train MAE   30.57s | val MAE   30.05s
epoch  14 | train MAE   30.59s | val MAE   30.27s
epoch  15 | train MAE   30.61s | val MAE   29.99s
epoch  16 | train MAE   30.66s | val MAE   30.09s
epoch  17 | train MAE   30.73s | val MAE   30.31s
epoch  18 | train MAE   30.84s | val MAE   30.40s
epoch  19 | train MAE   30.95s | val MAE   30.86s
epoch  20 | train MAE   31.08s | val MAE   30.35s


<All keys matched successfully>

In [19]:
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for cat, num, target, lengths in test_loader:
        cat, num = cat.to(DEVICE), num.to(DEVICE)
        pred = model(cat, num, lengths).cpu()
        for i, length in enumerate(lengths):
            all_preds.append(pred[i, :length].numpy())
            all_targets.append(target[i, :length].numpy())
 
preds = np.concatenate(all_preds)
targets = np.concatenate(all_targets)
resid = preds - targets
 
mae = np.abs(resid).mean()
rmse = np.sqrt((resid ** 2).mean())
median_ae = np.median(np.abs(resid))
bias = resid.mean()
ss_res = (resid ** 2).sum()
ss_tot = ((targets - targets.mean()) ** 2).sum()
r2 = 1 - ss_res / ss_tot
 
print(f"\nMAE:       {mae:.1f} sec")
print(f"Median AE: {median_ae:.1f} sec")
print(f"RMSE:      {rmse:.1f} sec")
print(f"Bias:      {bias:+.1f} sec")
print(f"R2:        {r2:.4f}")
for thresh in (30, 60, 120):
    print(f"within {thresh}s: {(np.abs(resid) <= thresh).mean():.1%}")
 
torch.save(
    {"model_state": model.state_dict(), "cat_maps": cat_maps, "num_means": num_means, "num_stds": num_stds},
    "models/gru_baseline.pt",
)
print("\nmodel saved to models/gru_baseline.pt")


MAE:       29.9 sec
Median AE: 20.4 sec
RMSE:      54.6 sec
Bias:      -4.0 sec
R2:        0.5016
within 30s: 65.8%
within 60s: 90.2%
within 120s: 97.8%

model saved to models/gru_baseline.pt
